# 02 — Pré-processamento dos Dados

**Tech Challenge Fase 02 — Sistema de Recomendação (Instacart)**

Prepara os dados brutos para a modelagem, gerando os artefatos que os próximos
notebooks (features, baselines, MLP) irão consumir. Corresponde ao estágio
`preprocess` do futuro pipeline DVC.

**Saídas (em `data/processed/`):**
- `interactions.parquet` — interações únicas (usuário, produto) do histórico.
- `user_id_map.parquet` / `product_id_map.parquet` — códigos contíguos p/ embeddings.
- `val_ground_truth.parquet` — cestas reais do próximo pedido (para avaliação).

As células seguem o padrão de **funções reutilizáveis**, para facilitar a futura
migração para `src/`.

## 0. Setup

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)


def find_project_root(marker: str = 'pyproject.toml') -> Path:
    '''Sobe na arvore de diretorios ate encontrar o marcador do projeto.'''
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f'Marcador {marker} nao encontrado a partir de {current}')


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Raw      :', RAW_DIR)
print('Processed:', PROCESSED_DIR)

## 1. Carregar os dados necessários

Carregamos apenas o que o pré-processamento precisa (`orders`, `prior`, `train`),
com tipos reduzidos para economizar memória.

In [ ]:
def load_orders(raw_dir: Path) -> pd.DataFrame:
    '''Carrega a tabela de pedidos com tipos economicos de memoria.'''
    dtypes = {
        'order_id': 'int32',
        'user_id': 'int32',
        'eval_set': 'category',
        'order_number': 'int16',
        'order_dow': 'int8',
        'order_hour_of_day': 'int8',
        'days_since_prior_order': 'float32',
    }
    return pd.read_csv(raw_dir / 'orders.csv', dtype=dtypes)


def load_order_products(raw_dir: Path, name: str) -> pd.DataFrame:
    '''Carrega um arquivo order_products ('prior' ou 'train') com tipos reduzidos.'''
    dtypes = {
        'order_id': 'int32',
        'product_id': 'int32',
        'add_to_cart_order': 'int16',
        'reordered': 'int8',
    }
    return pd.read_csv(raw_dir / f'order_products__{name}.csv', dtype=dtypes)


orders = load_orders(RAW_DIR)
prior = load_order_products(RAW_DIR, 'prior')
order_products_train = load_order_products(RAW_DIR, 'train')
print('orders:', orders.shape)
print('prior :', prior.shape)
print('train :', order_products_train.shape)

## 2. Tabela de interações (usuário × produto)

A unidade de aprendizado do recomendador é o par (usuário, produto). O `user_id`
está em `orders` e os produtos em `order_products__prior`, então juntamos pelo
`order_id` e agregamos o histórico em interações únicas, guardando a **força** da
interação: quantas vezes o usuário comprou o produto (`n_orders`) e quantas foram
recompra (`n_reorders`).

In [ ]:
def build_interactions(prior: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    '''Agrega o historico (prior) em interacoes unicas (usuario, produto).'''
    merged = prior.merge(
        orders[['order_id', 'user_id']], on='order_id', how='left'
    )
    interactions = (
        merged.groupby(['user_id', 'product_id'])
        .agg(n_orders=('reordered', 'size'), n_reorders=('reordered', 'sum'))
        .reset_index()
    )
    interactions['n_orders'] = interactions['n_orders'].astype('int32')
    interactions['n_reorders'] = interactions['n_reorders'].astype('int32')
    return interactions


interactions = build_interactions(prior, orders)
print('Interações únicas (user, product):', f'{len(interactions):,}')
interactions.head()

## 3. Codificar IDs em índices contíguos

Camadas de *embedding* exigem índices inteiros contíguos (0…N-1). Os ids
originais são esparsos (chegam a dezenas de milhares, com buracos), então criamos
um mapa `id → idx` para usuários e produtos. Os mapas são salvos para reaplicar
na validação e para decodificar as previsões mais tarde.

In [ ]:
def make_encoder(values: pd.Series) -> pd.Series:
    '''Mapeia cada id unico para um indice contiguo (ordenado e deterministico).'''
    uniques = np.sort(values.unique())
    return pd.Series(np.arange(len(uniques), dtype='int32'), index=uniques)


user_encoder = make_encoder(interactions['user_id'])
product_encoder = make_encoder(interactions['product_id'])

interactions['user_idx'] = interactions['user_id'].map(user_encoder).astype('int32')
interactions['product_idx'] = (
    interactions['product_id'].map(product_encoder).astype('int32')
)

print('Usuários :', f'{len(user_encoder):,}')
print('Produtos :', f'{len(product_encoder):,}')
interactions.head()

## 4. Conjunto de validação — o "próximo pedido"

Para cada usuário do `eval_set = train`, o pedido `train` é o próximo pedido real
(o que queremos prever). Montamos a verdade-base (*ground truth*): os produtos
efetivamente comprados nesse pedido. Produtos nunca vistos no histórico não têm
índice (*cold-start*) — contamos quantos são.

In [ ]:
def build_validation(
    orders: pd.DataFrame,
    order_products_train: pd.DataFrame,
    user_encoder: pd.Series,
    product_encoder: pd.Series,
) -> pd.DataFrame:
    '''Monta a verdade-base do proximo pedido para usuarios do eval_set 'train'.'''
    train_orders = orders.loc[
        orders['eval_set'] == 'train', ['order_id', 'user_id']
    ]
    ground_truth = order_products_train.merge(
        train_orders, on='order_id', how='inner'
    )
    ground_truth = ground_truth[['user_id', 'product_id', 'reordered']].copy()
    ground_truth['user_idx'] = (
        ground_truth['user_id'].map(user_encoder).astype('Int32')
    )
    ground_truth['product_idx'] = (
        ground_truth['product_id'].map(product_encoder).astype('Int32')
    )
    return ground_truth


val_ground_truth = build_validation(
    orders, order_products_train, user_encoder, product_encoder
)
n_cold = val_ground_truth['product_idx'].isna().sum()
print('Linhas de validação       :', f'{len(val_ground_truth):,}')
print('Usuários na validação     :', f"{val_ground_truth['user_id'].nunique():,}")
print('Itens cold-start (sem idx):', f'{n_cold:,}')
val_ground_truth.head()

## 5. Salvar artefatos processados

Salvamos em `data/processed/` (fora do git; será versionado pelo DVC na Etapa 3).
O formato *parquet* é compacto e preserva os tipos das colunas.

In [ ]:
def save_parquet(df: pd.DataFrame, path: Path) -> None:
    '''Salva um DataFrame em parquet e informa o resultado.'''
    df.to_parquet(path, index=False)
    print(f'  saved {path.name}  ({len(df):,} linhas)')


user_map = user_encoder.rename_axis('user_id').reset_index(name='user_idx')
product_map = product_encoder.rename_axis('product_id').reset_index(
    name='product_idx'
)

save_parquet(interactions, PROCESSED_DIR / 'interactions.parquet')
save_parquet(user_map, PROCESSED_DIR / 'user_id_map.parquet')
save_parquet(product_map, PROCESSED_DIR / 'product_id_map.parquet')
save_parquet(val_ground_truth, PROCESSED_DIR / 'val_ground_truth.parquet')

## 6. Resumo

- **`interactions.parquet`** — interações únicas (user, product) com força
  (`n_orders`, `n_reorders`) e os índices para embeddings.
- **`user_id_map` / `product_id_map`** — tradução id ↔ índice contíguo.
- **`val_ground_truth.parquet`** — cestas reais do próximo pedido, para avaliar
  o ranking do modelo.

**Backlog de refatoração (para `src/` na fase final):**
`find_project_root`, `load_orders`, `load_order_products` → `src/data/`;
`build_interactions`, `make_encoder`, `build_validation` → `src/data/` ou
`src/features/`. Cada função aqui já é candidata direta a virar parte de um
módulo + um stage do DVC.

**Próximo notebook (`03`):** engenharia de features e/ou definição do *framing*
de treino (NCF com amostragem de negativos vs. predição de recompra).